In [1]:
# ============================================================================
# 05_power_and_design.ipynb
# Pre-experiment design check. Variance of the content-distance measure is
# estimated from the generation pilot, three variance scenarios are built
# around it, and a Monte-Carlo simulation establishes, for a range of sample
# sizes, (i) the half-width of the repeat-baseline confidence interval,
# (ii) the half-width of the delta interval, and (iii) the power to detect an
# additional variation of delta = 0.10. The resulting design recommendation is
# written for the generation stage. No LLM calls are made.
# ============================================================================

# [Imports and global configuration]
import json
import warnings
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
SEED = 42
rng = np.random.RandomState(SEED)

ROOT = Path("..")
DATA_DIR = ROOT / "data"
REF_DIR = DATA_DIR / "reference"
FIG_DIR = ROOT / "results" / "figures"
TAB_DIR = ROOT / "results" / "tables"
for d in (FIG_DIR, TAB_DIR):
    d.mkdir(parents=True, exist_ok=True)

# [Plotting style: grayscale, publication settings, no captions]
sns.set_theme(style="whitegrid", context="paper")
sns.set_palette(sns.color_palette("gray", n_colors=8))
plt.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 600, "font.family": "sans-serif",
    "axes.grid": True, "grid.color": "0.85", "grid.linewidth": 0.5,
    "axes.edgecolor": "0.3", "axes.labelcolor": "0.15", "text.color": "0.15",
    "xtick.color": "0.3", "ytick.color": "0.3", "image.cmap": "gray",
    "savefig.bbox": "tight", "savefig.transparent": False,
})

def save_fig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(FIG_DIR / f"{name}.{ext}", dpi=600, bbox_inches="tight")
    plt.close(fig)
    print(f"saved figure: {name}")

def save_table(df, name, index=False):
    df.to_csv(TAB_DIR / f"{name}.csv", index=index, encoding="utf-8-sig")
    print(f"saved table: {name}")

# [Estimate content-distance variance from the generation pilot]
# Feature distance between two explanations = 1 - Jaccard of their top-k feature
# sets. The pilot provides several explanations per (domain, model, case); the
# spread of these within-case distances anchors the simulation.
with open(REF_DIR / "pilot_extractions.json", encoding="utf-8") as f:
    pilot = json.load(f)
pdf = pd.DataFrame([{"domain": r["domain"], "model": r["model"], "case_id": r["case_id"],
                     "feats": frozenset(c["feature"] for c in r["claims"]
                                        if c["feature"] != "OUT_OF_LIST")}
                    for r in pilot])
dists = []
for _, grp in pdf.groupby(["domain", "model", "case_id"]):
    sets = list(grp["feats"])
    for a, b in combinations(sets, 2):
        dists.append(1 - (len(a & b) / len(a | b) if (a | b) else 1.0))
dists = np.array(dists)
PILOT_MEAN, PILOT_SD = float(dists.mean()), float(dists.std())
print(f"pilot feature-distance: mean={PILOT_MEAN:.4f} sd={PILOT_SD:.4f} (n={len(dists)})")

# [Design targets and variance scenarios]
TARGETS = {"baseline_ci_halfwidth": 0.05, "delta_ci_halfwidth": 0.05,
           "delta_detect": 0.10, "power": 0.80}
SCENARIOS = {
    "optimistic": dict(base_mean=PILOT_MEAN * 0.6, case_sd=PILOT_SD * 0.6),
    "pilot": dict(base_mean=PILOT_MEAN, case_sd=PILOT_SD),
    "pessimistic": dict(base_mean=PILOT_MEAN * 1.4, case_sd=PILOT_SD * 1.3),
}
N_GRID = [60, 100, 150, 200, 300, 400]
N_BOOT, N_SIM = 400, 300
PLANNED = {"n_cases_main": 300, "n_repeats_main": 3, "n_cases_deep": 60,
           "n_repeats_deep": 20, "top_k": 5}

def clip01(x):
    return np.clip(x, 0.0, 1.0)

def simulate(n_cases, sc, detect):
    # One synthetic experiment: case-level baselines and a changed condition
    # offset by `detect`. Returns (baseline mean CI halfwidth, delta CI
    # halfwidth, delta interval excludes zero).
    base = clip01(rng.normal(sc["base_mean"], sc["case_sd"], n_cases))
    changed = clip01(base + detect + rng.normal(0, sc["case_sd"] * 0.5, n_cases))
    delta_case = changed - base
    # Cluster (case-level) bootstrap.
    bmeans, dmeans = np.empty(N_BOOT), np.empty(N_BOOT)
    for b in range(N_BOOT):
        idx = rng.randint(0, n_cases, n_cases)
        bmeans[b] = base[idx].mean()
        dmeans[b] = delta_case[idx].mean()
    bhw = (np.percentile(bmeans, 97.5) - np.percentile(bmeans, 2.5)) / 2
    dhw = (np.percentile(dmeans, 97.5) - np.percentile(dmeans, 2.5)) / 2
    excl0 = np.percentile(dmeans, 2.5) > 0
    return bhw, dhw, excl0

# [Run the simulation over the sample-size grid and scenarios]
grid_rows = []
for name, sc in SCENARIOS.items():
    for n in N_GRID:
        bhws, dhws, powers = [], [], []
        for _ in range(N_SIM):
            bhw, dhw, excl0 = simulate(n, sc, TARGETS["delta_detect"])
            bhws.append(bhw); dhws.append(dhw); powers.append(excl0)
        grid_rows.append({"scenario": name, "n_cases": n,
                          "baseline_ci_halfwidth": round(np.mean(bhws), 4),
                          "delta_ci_halfwidth": round(np.mean(dhws), 4),
                          "power_delta_010": round(np.mean(powers), 3)})
grid = pd.DataFrame(grid_rows)
save_table(grid, "tab06_power_grid")

# [Verdicts at the planned sample size]
verdicts = []
for name in SCENARIOS:
    row = grid[(grid.scenario == name) & (grid.n_cases == PLANNED["n_cases_main"])].iloc[0]
    verdicts.append({
        "scenario": name,
        "baseline_ci_halfwidth": row["baseline_ci_halfwidth"],
        "baseline_target_met": bool(row["baseline_ci_halfwidth"] <= TARGETS["baseline_ci_halfwidth"]),
        "delta_ci_halfwidth": row["delta_ci_halfwidth"],
        "delta_precision_met": bool(row["delta_ci_halfwidth"] <= TARGETS["delta_ci_halfwidth"]),
        "power_at_010": row["power_delta_010"],
        "power_target_met": bool(row["power_delta_010"] >= TARGETS["power"]),
    })
vdf = pd.DataFrame(verdicts)
save_table(vdf, "tab07_design_verdicts")
design_adequate = bool(vdf[["baseline_target_met", "delta_precision_met", "power_target_met"]].all().all())

# Minimum cases to reach the power target, per scenario.
min_req = []
for name in SCENARIOS:
    ok = grid[(grid.scenario == name) & (grid.power_delta_010 >= TARGETS["power"])]
    min_req.append({"scenario": name,
                    "min_cases_for_power": int(ok["n_cases"].min()) if len(ok) else None,
                    "planned_cases": PLANNED["n_cases_main"]})

design = {"planned": PLANNED, "targets": TARGETS,
          "pilot_structure": {"feature_distance_mean": round(PILOT_MEAN, 4),
                              "feature_distance_sd": round(PILOT_SD, 4),
                              "n_pairs": int(len(dists))},
          "scenarios": {k: {kk: round(vv, 4) for kk, vv in v.items()} for k, v in SCENARIOS.items()},
          "verdicts": verdicts, "minimum_requirements": min_req,
          "design_adequate": design_adequate,
          "final": PLANNED}
with open(DATA_DIR / "design_recommendation.json", "w", encoding="utf-8") as f:
    json.dump(design, f, indent=1, ensure_ascii=False)
print("saved design_recommendation.json | design adequate:", design_adequate)

# [Figure 16: baseline CI half-width vs sample size]
fig, ax = plt.subplots(figsize=(5.4, 3.6))
styles = {"optimistic": ":", "pilot": "-", "pessimistic": "--"}
shades = {"optimistic": "0.6", "pilot": "0.2", "pessimistic": "0.45"}
for name in SCENARIOS:
    sub = grid[grid.scenario == name]
    ax.plot(sub["n_cases"], sub["baseline_ci_halfwidth"], marker="o", ms=3,
            linestyle=styles[name], color=shades[name], label=name)
ax.axhline(TARGETS["baseline_ci_halfwidth"], color="0.1", lw=0.9, linestyle="-")
ax.set_xlabel("Cases"); ax.set_ylabel("Baseline CI half-width")
ax.legend(fontsize=7, frameon=False)
save_fig(fig, "fig16_baseline_precision")

# [Figure 17: power to detect delta = 0.10 vs sample size]
fig, ax = plt.subplots(figsize=(5.4, 3.6))
for name in SCENARIOS:
    sub = grid[grid.scenario == name]
    ax.plot(sub["n_cases"], sub["power_delta_010"], marker="o", ms=3,
            linestyle=styles[name], color=shades[name], label=name)
ax.axhline(TARGETS["power"], color="0.1", lw=0.9, linestyle="-")
ax.axvline(PLANNED["n_cases_main"], color="0.75", lw=0.8)
ax.set_xlabel("Cases"); ax.set_ylabel("Power at delta = 0.10"); ax.set_ylim(0, 1.05)
ax.legend(fontsize=7, frameon=False, loc="lower right")
save_fig(fig, "fig17_delta_power_curves")

# [Figure 18: estimator calibration - recovered delta vs true delta (pilot scenario)]
true_deltas = np.linspace(0, 0.4, 9)
est_mean, est_lo, est_hi = [], [], []
for td in true_deltas:
    ests = []
    for _ in range(120):
        base = clip01(rng.normal(SCENARIOS["pilot"]["base_mean"], SCENARIOS["pilot"]["case_sd"], 300))
        changed = clip01(base + td + rng.normal(0, SCENARIOS["pilot"]["case_sd"] * 0.5, 300))
        ests.append((changed - base).mean())
    ests = np.array(ests)
    est_mean.append(ests.mean()); est_lo.append(np.percentile(ests, 2.5)); est_hi.append(np.percentile(ests, 97.5))
fig, ax = plt.subplots(figsize=(4.6, 4.4))
ax.plot([0, 0.4], [0, 0.4], color="0.8", lw=0.9)
ax.plot(true_deltas, est_mean, marker="o", ms=3, color="0.2", lw=1.3)
ax.fill_between(true_deltas, est_lo, est_hi, color="0.75", alpha=0.5)
ax.set_xlabel("True additional variation"); ax.set_ylabel("Recovered delta")
save_fig(fig, "fig18_estimator_calibration")

# [Figure 19: simulated case-level delta distribution at the planned design]
base = clip01(rng.normal(SCENARIOS["pilot"]["base_mean"], SCENARIOS["pilot"]["case_sd"], 300))
changed = clip01(base + TARGETS["delta_detect"] + rng.normal(0, SCENARIOS["pilot"]["case_sd"] * 0.5, 300))
fig, ax = plt.subplots(figsize=(5.4, 3.4))
ax.hist(changed - base, bins=30, color="0.5", edgecolor="0.15")
ax.axvline((changed - base).mean(), color="0.1", lw=1.1, linestyle="--")
ax.set_xlabel("Case-level delta (changed - baseline)"); ax.set_yticks([])
save_fig(fig, "fig19_case_level_delta_distribution")

print("\nDesign check complete. Planned:", PLANNED["n_cases_main"],
      "cases; design adequate:", design_adequate)
print(vdf.to_string(index=False))


pilot feature-distance: mean=0.4220 sd=0.3589 (n=900)


saved table: tab06_power_grid
saved table: tab07_design_verdicts
saved design_recommendation.json | design adequate: True


saved figure: fig16_baseline_precision
saved figure: fig17_delta_power_curves


saved figure: fig18_estimator_calibration
saved figure: fig19_case_level_delta_distribution

Design check complete. Planned: 300 cases; design adequate: True
   scenario  baseline_ci_halfwidth  baseline_target_met  delta_ci_halfwidth  delta_precision_met  power_at_010  power_target_met
 optimistic                 0.0217                 True              0.0116                 True           1.0              True
      pilot                 0.0339                 True              0.0181                 True           1.0              True
pessimistic                 0.0385                 True              0.0215                 True           1.0              True
